# BERT — Bidirectional Encoder Representations from Transformers

**Course:** Artificial Intelligence · Unit 5 — Machine Learning  
**Institution:** Universidad Tecnológica de Bolívar  
**Reference:** Russell, S. & Norvig, P. (2022). *AIMA* (4th ed.), Ch. 24; Devlin et al. (2019). *BERT: Pre-training of Deep Bidirectional Transformers*  
**Python compatibility:** 3.10 · 3.11 · 3.12 · 3.13

---


# Question-Answering

In [26]:
# pip install transformers  # run once
# pip install pytorch  # run once


## Data Directory Setup


In [1]:
from pathlib import Path

DIR_DATA = Path.cwd() / 'data'
DIR_DATA


'C:\\Users\\epuerta\\OneDrive - Universidad Tecnológica de Bolívar\\Apps\\AppsISCO\\InteligenciaArtificial-ISCO-A06A\\laboratories\\data\\'

In [3]:
import nltk
# Download the required NLTK sentence-tokeniser data
nltk.download('punkt_tab', quiet=True)


In [4]:
filename = DIR_DATA / 'AboutMe.txt'
filename


'C:\\Users\\epuerta\\OneDrive - Universidad Tecnológica de Bolívar\\Apps\\AppsISCO\\InteligenciaArtificial-ISCO-A06A\\laboratories\\data\\AboutMe.txt'

In [5]:
# Open the file in read mode
with open(filename, 'r') as f:
    context = f.read()

# Display the file content
print(context)


Edwin Puertas is an AI Software Architect and NLP Researcher. Currently, I am an Associate Professor at the Universidad TecnolÃ³gica de BolÃ­var, and I simultaneously serve as a bridge between academia and industry in projects related to AI. My work focuses on developing innovative AI solutions that bridge the gap between theoretical research and practical applications. With over 19 years of experience in academia, I have taught courses in Artificial Intelligence, Natural Language Processing, Big Data, and Software Engineering, fostering an environment of collaborative learning and research excellence.

As an AI Software Architect, I lead projects involving the design and implementation of scalable AI systems, applying advanced machine learning techniques to solve complex problems across various industries. My research in NLP explores the intricacies of human language, aiming to enhance natural language understanding and develop applications that improve human-computer interaction.

I 

> **Output interpretation:** The `AboutMe.txt` passage serves as the QA context. Each sentence will be tokenised and queried by BERT to extract an answer span.


In [6]:
from transformers import AutoTokenizer, AutoModelForQuestionAnswering, pipeline
the_model = 'mrm8488/distill-bert-base-spanish-wwm-cased-finetuned-spa-squad2-es'
tokenizer = AutoTokenizer.from_pretrained(the_model, do_lower_case=False)
model = AutoModelForQuestionAnswering.from_pretrained(the_model)


Some weights of the model checkpoint at mrm8488/distill-bert-base-spanish-wwm-cased-finetuned-spa-squad2-es were not used when initializing BertForQuestionAnswering: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
- This IS expected if you are initializing BertForQuestionAnswering from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForQuestionAnswering from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


> **Output interpretation:** The model loads from a Spanish DistilBERT checkpoint fine-tuned on SQuAD2-ES. The pooler-weight warning is expected: the pooler head is unused in span-extraction QA.


In [11]:
print(model)

BertForQuestionAnswering(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(31002, 768, padding_idx=1)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12, 

> **Output interpretation:** The architecture has 12 BERT encoder layers (768 hidden units, 12 attention heads). The `qa_outputs` linear head maps encoder states to two logits per token: answer-start and answer-end positions.


In [12]:
question = "¿Cuantos años de experiencia tiene?"

# Tokenise the context into sentences using NLTK
context_sentences = nltk.sent_tokenize(context)

# Prepend the question so it is also tokenised
sentences = [question] + context_sentences

# Process each sentence individually
for idx, sentence in enumerate(sentences):
    # Encode with special tokens
    encoding = tokenizer.encode_plus(
        sentence, return_tensors='pt', add_special_tokens=True
    )

    # Convert input IDs to readable tokens
    input_ids = encoding['input_ids']
    tokens = tokenizer.convert_ids_to_tokens(input_ids[0])

    # Display the original sentence and its BERT tokens
    print(f'Sentence {idx + 1}:')
    print(f'Original text: {sentence}')
    print(f'BERT tokens:   {tokens}')
    print('-' * 50)  # Separator


Oración 1:
Texto original: ¿Cuantos años de experiencia tiene?
Tokens de BERT: ['[CLS]', '¿', 'Cuanto', '##s', 'años', 'de', 'experiencia', 'tiene', '?', '[SEP]']
--------------------------------------------------
Oración 2:
Texto original: Edwin Puertas is an AI Software Architect and NLP Researcher.
Tokens de BERT: ['[CLS]', '[UNK]', 'Puerta', '##s', 'is', 'an', 'AI', '[UNK]', 'Archi', '##tec', '##t', 'and', 'NL', '##P', 'Research', '##er', '.', '[SEP]']
--------------------------------------------------
Oración 3:
Texto original: Currently, I am an Associate Professor at the Universidad TecnolÃ³gica de BolÃ­var, and I simultaneously serve as a bridge between academia and industry in projects related to AI.
Tokens de BERT: ['[CLS]', 'Cur', '##ren', '##t', '##ly', ',', 'I', 'am', 'an', 'Associa', '##te', 'Profes', '##sor', 'at', 'the', 'Universidad', '[UNK]', 'de', '[UNK]', ',', 'and', 'I', 'simul', '##tan', '##eo', '##us', '##ly', 'serv', '##e', 'as', 'a', 'bri', '##dge', '[UNK]', 'a

> **Output interpretation:** WordPiece splits words into subword units prefixed with `##`. Characters outside the Spanish vocabulary appear as `[UNK]`. Every sequence is wrapped with `[CLS]` (start) and `[SEP]` (end) tokens.


In [13]:
# Question-answering pipeline example
nlp = pipeline('question-answering', model=model, tokenizer=tokenizer)
result = nlp({'question': question, 'context': context})
print(result)


{'score': 0.24410812556743622, 'start': 378, 'end': 385, 'answer': 'over 19'}


> **Output interpretation:** The pipeline returns `answer`, `score` (confidence), and character `start`/`end` offsets within the context. A score of ~0.24 indicates moderate confidence in the extracted span.


In [14]:
from textwrap import wrap


def question_answer(model, context, nlp):
    """Run an interactive QA session against the given context.

    Args:
        model: The pre-trained BERT model.
        context (str): The passage of text to query.
        nlp: The Hugging Face question-answering pipeline.
    """
    # Print the context
    print('Context:')
    print('-----------------')
    print('\n'.join(wrap(context)))

    # Interactive loop (empty input exits)
    running = True
    while running:
        print('\nQuestion:')
        print('-----------------')
        question = str(input())

        running = question != ''

        if running:
            result = nlp({'question': question, 'context': context})
            print('\nAnswer:')
            print('-----------------')
            print(result['answer'])


In [15]:
question_answer(model, context, nlp)


Contexto:
-----------------
Edwin Puertas is an AI Software Architect and NLP Researcher.
Currently, I am an Associate Professor at the Universidad TecnolÃ³gica
de BolÃ­var, and I simultaneously serve as a bridge between academia
and industry in projects related to AI. My work focuses on developing
innovative AI solutions that bridge the gap between theoretical
research and practical applications. With over 19 years of experience
in academia, I have taught courses in Artificial Intelligence, Natural
Language Processing, Big Data, and Software Engineering, fostering an
environment of collaborative learning and research excellence.  As an
AI Software Architect, I lead projects involving the design and
implementation of scalable AI systems, applying advanced machine
learning techniques to solve complex problems across various
industries. My research in NLP explores the intricacies of human
language, aiming to enhance natural language understanding and develop
applications that improve hum

 ¿donde estudio?



Respuesta:
-----------------
Universidad TecnolÃ³gica de BolÃ­var

Pregunta:
-----------------


KeyboardInterrupt: Interrupted by user

In [ ]:
context = "Avengers: Endgame (Vengadores: Endgame en España)2​ es una película de superhéroes estadounidense de 2019 basada en el equipo de superhéroes Los Vengadores de Marvel Comics. Producida por Marvel Studios y distribuida por Walt Disney Studios Motion Pictures, es la secuela directa de Avengers: Infinity War (2018) y la película número 22 del Universo cinematográfico de Marvel (UCM), y hasta la fecha la más larga. Dirigida por Anthony y Joe Russo y escrita por Christopher Markus y Stephen McFeely, la película cuenta con un reparto coral que incluye a Robert Downey Jr., Chris Evans, Mark Ruffalo, Chris Hemsworth, Scarlett Johansson, Jeremy Renner, Don Cheadle, Paul Rudd, Brie Larson, Karen Gillan, Danai Gurira, Benedict Wong, Jon Favreau, Bradley Cooper, Gwyneth Paltrow y Josh Brolin. En la película, los miembros supervivientes de Los Vengadores y sus aliados intentan revertir el daño causado por Thanos en Infinity War."
question_answer(model, context, nlp)


Contexto:
-----------------
Avengers: Endgame (Vengadores: Endgame en España)2​ es una película de
superhéroes estadounidense de 2019 basada en el equipo de superhéroes
Los Vengadores de Marvel Comics. Producida por Marvel Studios y
distribuida por Walt Disney Studios Motion Pictures, es la secuela
directa de Avengers: Infinity War (2018) y la película número 22 del
Universo cinematográfico de Marvel (UCM), y hasta la fecha la más
larga. Dirigida por Anthony y Joe Russo y escrita por Christopher
Markus y Stephen McFeely, la película cuenta con un reparto coral que
incluye a Robert Downey Jr., Chris Evans, Mark Ruffalo, Chris
Hemsworth, Scarlett Johansson, Jeremy Renner, Don Cheadle, Paul Rudd,
Brie Larson, Karen Gillan, Danai Gurira, Benedict Wong, Jon Favreau,
Bradley Cooper, Gwyneth Paltrow y Josh Brolin. En la película, los
miembros supervivientes de Los Vengadores y sus aliados intentan
revertir el daño causado por Thanos en Infinity War.

Pregunta:
-----------------


 ¿quien Chris evans?



Respuesta:
-----------------
de Marvel (UCM), y hasta la

Pregunta:
-----------------
